In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\Users\luis\Documents\TFG - Data-Centric AI")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
import pandas as pd
from IPython.display import display

from utils.thresholds.quality import (
    ensure_quality_metric,
    evaluate_quality_thresholds_against_manual_labels,
    export_quality_review_images,
    get_quality_filter_spec,
    recommend_quality_threshold,
    sample_images_for_quality_threshold_review,
)

FILTER_NAME = "darkness"
METADATA_PATH = PROJECT_ROOT / "data" / "phase2" / "phase2_train.csv"
IMAGES_DIR = PROJECT_ROOT / "data" / "phase2" / "frames"

REPORTS_DIR = PROJECT_ROOT / "reports" / f"{FILTER_NAME}_threshold_selection"
REVIEW_IMAGES_DIR = REPORTS_DIR / "review_images"
REVIEW_CSV_PATH = REPORTS_DIR / "manual_quality_review.csv"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

spec = get_quality_filter_spec(FILTER_NAME)
METRIC_COLUMN = spec["metric_column"]
THRESHOLD_CANDIDATES = spec["thresholds"]
N_REVIEW_IMAGES = 100
N_REVIEW_IMAGES_PER_THRESHOLD = N_REVIEW_IMAGES // len(THRESHOLD_CANDIDATES)

print("FILTER_NAME:", FILTER_NAME)
print("METADATA_PATH:", METADATA_PATH)
print("IMAGES_DIR:", IMAGES_DIR)
print("THRESHOLD_CANDIDATES:", THRESHOLD_CANDIDATES)

FILTER_NAME: darkness
METADATA_PATH: C:\Users\luis\Documents\TFG - Data-Centric AI\data\phase2\phase2_train.csv
IMAGES_DIR: C:\Users\luis\Documents\TFG - Data-Centric AI\data\phase2\frames
THRESHOLD_CANDIDATES: [30.0, 35.0, 40.0, 45.0, 50.0]


In [3]:
df = pd.read_csv(METADATA_PATH)
df = ensure_quality_metric(
    dataframe=df,
    filter_name=FILTER_NAME,
    images_dir=IMAGES_DIR,
)

print("Images:", len(df))
display(df[["filename", METRIC_COLUMN]].head())
display(df[METRIC_COLUMN].describe())

Images: 7978


,filename,brightness_v_mean
0,20241204_134604_R0_F0_S1_8f99b423ac93fa5b.jpg,116.906393
1,20241213_102820_R0_F0_S1_34b16ba5d8336515.jpg,84.247481
2,20241213_102821_R2_F0_S1_34b16ba5d8336515.jpg,76.631638
3,20241213_103001_R2_F0_S8_34b16ba5d8336515.jpg,131.618293
4,20241213_103008_R2_F0_S9_34b16ba5d8336515.jpg,98.607903


count    7978.000000
mean      126.871106
std        32.911786
min        13.505730
25%       103.113093
50%       131.862804
75%       152.232311
max       229.815524
Name: brightness_v_mean, dtype: float64

In [4]:
review_df = sample_images_for_quality_threshold_review(
    dataframe=df,
    filter_name=FILTER_NAME,
    thresholds=THRESHOLD_CANDIDATES,
    n_per_threshold=N_REVIEW_IMAGES_PER_THRESHOLD,
)

review_df = export_quality_review_images(
    review_df=review_df,
    images_dir=IMAGES_DIR,
    output_dir=REVIEW_IMAGES_DIR,
    image_size=(320, 180),
)

review_df.to_csv(REVIEW_CSV_PATH, index=False)

print("CSV to label:", REVIEW_CSV_PATH)
print("Images exported to:", REVIEW_IMAGES_DIR)
display(review_df.head())

CSV to label: C:\Users\luis\Documents\TFG - Data-Centric AI\reports\darkness_threshold_selection\manual_quality_review.csv
Images exported to: C:\Users\luis\Documents\TFG - Data-Centric AI\reports\darkness_threshold_selection\review_images


,patient_id,day,R,F,hour,histology,filename,video_filename,elapsed_seconds,crop_x,...,detection_confidence,bbox_area_ratio,brightness_v_mean,filter_name,threshold,predicted_discard,distance_to_threshold,manual_label,manual_comment,review_image_path
0,5758749,20250204,R1,F0,123919.0,Sessile_serrated_adenoma,20250204_123919_R1_F0_S4_b84d97be31ae6df9.jpg,20250204_123250_R1_b84d97be31ae6df9.mp4,389.0,341,...,0.000000,0.000000,29.291461,darkness,30.0,True,0.708539,,,C:\Users\luis\Documents\TFG - Data-Centric AI\...
1,70079153,20241209,R5,F0,NaN,Sessile_serrated_adenoma,20241209_104231_R5_F0_S5_0df333d196a4d1ea_3.jpg,20241209_104220_R5_0df333d196a4d1ea.mp4,11.0,221,...,0.708984,0.352608,28.858093,darkness,30.0,True,1.141907,,,C:\Users\luis\Documents\TFG - Data-Centric AI\...
2,716128,20250414,R4,F0,111225.0,Hyperplastic,20250414_111225_R4_F0_S1_d585bb6b9977505f.jpg,20250414_111222_R4_d585bb6b9977505f.mp4,3.0,411,...,0.000000,0.000000,28.429918,darkness,30.0,True,1.570082,,,C:\Users\luis\Documents\TFG - Data-Centric AI\...
3,5758994,20250207,R10,F0,102227.0,Adenoma,20250207_102227_R10_F0_S5_7024c1054e6c5afe.jpg,20250207_102136_R10_7024c1054e6c5afe.mp4,51.0,120,...,0.602051,0.039397,28.256115,darkness,30.0,True,1.743885,,,C:\Users\luis\Documents\TFG - Data-Centric AI\...
4,5660304,20250127,R1,F0,NaN,Adenoma,20250127_092023_R1_F0_S4_448cf72de1c9771e_1.jpg,20250127_091956_R1_448cf72de1c9771e.mp4,27.0,211,...,0.559082,0.311789,27.830504,darkness,30.0,True,2.169496,,,C:\Users\luis\Documents\TFG - Data-Centric AI\...


In [4]:
labeled_df = pd.read_csv(REVIEW_CSV_PATH, sep=";", engine="python")
labeled_df["manual_label"] = (
    labeled_df["manual_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

valid_labels = {"discard", "valuable", "uncertain"}
invalid_labels = set(labeled_df["manual_label"].unique()) - valid_labels

if invalid_labels:
    raise ValueError(f"Invalid labels: {invalid_labels}")

label_counts = (
    labeled_df["manual_label"]
    .value_counts()
    .rename_axis("manual_label")
    .reset_index(name="count")
)

display(label_counts)

,manual_label,count
0,discard,51
1,valuable,49


In [5]:
manual_evaluation_df = evaluate_quality_thresholds_against_manual_labels(
    labeled_df=labeled_df,
    filter_name=FILTER_NAME,
    thresholds=THRESHOLD_CANDIDATES,
    positive_labels=("discard",),
    negative_labels=("valuable",),
)

manual_evaluation_df = manual_evaluation_df.sort_values(
    ["precision", "recall", "false_positive"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(manual_evaluation_df)

,filter_name,threshold,labelled_images,true_positive,false_positive,false_negative,true_negative,precision,recall
0,darkness,40.0,100,46,32,5,17,0.589744,0.901961
1,darkness,45.0,100,46,39,5,10,0.541176,0.901961
2,darkness,35.0,100,36,32,15,17,0.529412,0.705882
3,darkness,50.0,100,46,41,5,8,0.528736,0.901961
4,darkness,30.0,100,28,32,23,17,0.466667,0.549020


In [6]:
recommended_df = recommend_quality_threshold(
    evaluation_df=manual_evaluation_df,
    min_precision=0.80,
)

display(recommended_df)

selected = recommended_df.iloc[0]
FINAL_THRESHOLD = float(selected["threshold"])

print(f"Selected {FILTER_NAME} threshold: {FINAL_THRESHOLD:g}")

,filter_name,threshold,labelled_images,true_positive,false_positive,false_negative,true_negative,precision,recall
0,darkness,40.0,100,46,32,5,17,0.589744,0.901961
1,darkness,45.0,100,46,39,5,10,0.541176,0.901961
2,darkness,35.0,100,36,32,15,17,0.529412,0.705882
3,darkness,50.0,100,46,41,5,8,0.528736,0.901961
4,darkness,30.0,100,28,32,23,17,0.466667,0.549020


Selected darkness threshold: 40
